# 5-2 손실 함수 선택 심화

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
runs = {
    "R": {"task": "regression", "out": (4, 1), "target": (4, 1), "dtype": "long"},
    "B": {"task": "binary", "out": (4, 2), "target": (4,), "dtype": "long"},
    "M": {"task": "multiclass", "out": (4, 3), "target": (4,), "dtype": "float"},
}
violations = []
# 회귀는 output과 같은 shape의 연속값 target이 필요하므로 shape와 float dtype을 한 계약으로 봅니다.
if runs["R"]["out"] != runs["R"]["target"] or runs["R"]["dtype"] != "float":
    violations.append("R:float_target_required")
# 기본 이진 BCE와 다중 class CE는 target 표현이 다르므로 분류라는 이유로 같은 dtype을 강제하지 않습니다.
if runs["B"]["out"] != (4, 1) or runs["B"]["target"] != (4, 1) or runs["B"]["dtype"] != "float":
    violations.append("B:(B,1)_float_contract")
if runs["M"]["target"] != (4,) or runs["M"]["dtype"] != "long":
    violations.append("M:(B,)_long_contract")
print("violations:", violations)

violations: ['R:float_target_required', 'B:(B,1)_float_contract', 'M:(B,)_long_contract']


In [ ]:
# 검증 가능 정답 코드
import torch
from torch import nn


def compute_loss(task, output, target):
    # 회귀와 이진 태스크는 샘플당 값 하나인 (B,1) float 계약을 loss 호출 전에 명시적으로 검사합니다.
    if task == "regression":
        assert output.ndim == 2 and output.shape[1] == 1
        assert target.ndim == 2 and target.shape[1] == 1
        assert output.shape == target.shape
        assert output.dtype.is_floating_point and target.dtype.is_floating_point
        return nn.MSELoss()(output, target)
    if task == "binary":
        assert output.ndim == 2 and output.shape[1] == 1
        assert target.ndim == 2 and target.shape[1] == 1
        assert output.shape == target.shape
        assert output.dtype.is_floating_point and target.dtype.is_floating_point
        return nn.BCEWithLogitsLoss()(output, target)
    # 다중 class는 (B,C) raw logits와 범위가 유효한 (B,) long class index를 사용합니다.
    if task == "multiclass":
        assert output.ndim == 2 and output.shape[0] > 0 and output.shape[1] > 1
        assert output.dtype.is_floating_point
        assert target.ndim == 1 and target.shape == (output.shape[0],)
        assert target.dtype == torch.long
        assert int(target.min()) >= 0 and int(target.max()) < output.shape[1]
        return nn.CrossEntropyLoss()(output, target)
    raise ValueError("unknown task")

fixed_cases = [
    ("regression", torch.tensor([[1.], [3.]]), torch.tensor([[2.], [2.]])),
    ("binary", torch.tensor([[0.], [0.]]), torch.tensor([[0.], [1.]])),
    ("multiclass", torch.tensor([[2., 0.], [0., 2.]]), torch.tensor([0, 1])),
]
losses = [compute_loss(*case) for case in fixed_cases]
print("losses:", [round(v.item(), 4) for v in losses])
print("all_scalar:", all(v.ndim == 0 for v in losses))

In [2]:
# 검증 가능 정답 코드
targets = [[1, 0, 1], [0, 1, 0]]
# 한 행의 합이 1보다 큰 사례가 있으면 class index 하나로는 동시 태그 정보를 보존할 수 없습니다.
has_multiple = any(sum(row) > 1 for row in targets) # any(...) 구문은 하나라도 True가 있으면 True를 반환
candidates = {
    "A": {"supports_multiple": False, "loss": "CrossEntropyLoss"},
    "B": {"supports_multiple": True, "loss": "BCEWithLogitsLoss"},
}
# 문제에 주어진 독립 이진 태그 계약을 표현할 수 있는 후보만 남겨 loss 이름보다 label 의미를 우선합니다.
valid = [n for n, c in candidates.items() if c["supports_multiple"] == has_multiple]
print("has_multiple_labels:", has_multiple)
print("selected:", valid[0])
print("loss:", candidates[valid[0]]["loss"])

has_multiple_labels: True
selected: B
loss: BCEWithLogitsLoss
